# Step 12: Transformer trained from scratch
A Transformer encoder built in Keras and trained from scratch on the 25 frames of each second: each frame projected to 64 numbers, a learned positional encoding, 2 encoder blocks (4-head self-attention + feed-forward, each with Add & Norm), average pooling and a small dense head (development stage, random split).
*Same results as `src/12_transformer.py`.*

**In short** (what this notebook shows)

- **What:** A Transformer trained from scratch on the 25 frames of one second (positional encoding, 2 encoder blocks, attention).
- **Needs the NoXi data?** Yes. Random split (development step). The final 10-second Transformer is in notebook 29.
- **Result / use:** 0.841. Shows self-attention and the encoder block.

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: **24 conversations** (English 026, 027, 028, 029, 030, 058, 062, 063; French 001, 007, 009, 011, 013, 014, 020, 021; German 068, 070, 072, 073, 074, 075, 076, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

**Imports.**

In [3]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
keras.utils.set_random_seed(42)

In [4]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Data:** 25 frames x 802 features per second, random split 60 / 20 / 20, every frame scaled with a scaler fitted on training frames only.

In [5]:
y = pd.read_parquet("data/features/noxi_engagement.parquet", columns=["engagement"])["engagement"].to_numpy("float32")
X = np.load("data/features/noxi_sequences.npy", mmap_mode="r")   # read from disk piece by piece
rest, test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
train, val = train_test_split(rest, test_size=0.25, random_state=42)
scaler = StandardScaler().fit(X[train].reshape(-1, 802))


def prepare(rows):
    return scaler.transform(X[rows].reshape(-1, 802)).reshape(len(rows), 25, 802).astype("float32")


X_train, X_val, X_test = prepare(train), prepare(val), prepare(test)
y_train, y_val, y_test = y[train], y[val], y[test]
print("train", len(X_train), "| val", len(X_val), "| test", len(X_test))

train 29091 | val 9697 | test 9698


**Positional encoding:** attention does not know the order of the frames, so a learned vector is added for each of the 25 positions.

In [6]:
class PositionalEncoding(layers.Layer):
    def build(self, input_shape):
        self.pos = self.add_weight(shape=(25, 64), initializer="random_normal", name="pos")

    def call(self, x):
        return x + self.pos

**One encoder block:** self-attention (4 heads) and a feed-forward layer, each followed by Add & Norm.

In [7]:
def encoder_block(x):
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.1)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, attention]))
    ff = layers.Dense(128, activation="relu")(x)
    ff = layers.Dropout(0.1)(ff)
    ff = layers.Dense(64)(ff)
    return layers.LayerNormalization(epsilon=1e-6)(layers.Add()([x, ff]))

**The whole model**, top to bottom.

In [8]:
keras.utils.set_random_seed(42)
inp = layers.Input(shape=(25, 802))
x = layers.Dense(64)(inp)                 # each frame: 802 -> 64 numbers
x = PositionalEncoding()(x)               # add the position of each frame
x = encoder_block(x)                      # encoder block 1
x = encoder_block(x)                      # encoder block 2
x = layers.GlobalAveragePooling1D()(x)    # average the 25 frames
x = layers.Dropout(0.1)(x)
x = layers.Dense(64, activation="relu")(x)
out = layers.Dense(1)(x)                  # engagement
model = keras.Model(inp, out, name="Transformer")
model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
model.summary()

Model: "Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                 ┃ Output Shape              ┃         Param # ┃ Connected to              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)     │ (None, 25, 802)           │               0 │ -                         │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ dense (Dense)                │ (None, 25, 64)            │          51,392 │ input_layer[0][0]         │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ positional_encoding          │ (None, 25, 64)            │           1,600 │ dense[0][0]               │
│ (PositionalEncoding)         │                           │                 │                           │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ multi_head_attention         │ (None, 25, 64)            │          16,640 │ positional_encoding[0][0… │
│ (MultiHeadAttention)         │                           │                 │ positional_encoding[0][0] │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ add (Add)                    │ (None, 25, 64)            │               0 │ positional_encoding[0][0… │
│                              │                           │                 │ multi_head_attention[0][… │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ layer_normalization          │ (None, 25, 64)            │             128 │ add[0][0]                 │
│ (LayerNormalization)         │                           │                 │                           │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ dense_1 (Dense)              │ (None, 25, 128)           │           8,320 │ layer_normalization[0][0] │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ dropout_1 (Dropout)          │ (None, 25, 128)           │               0 │ dense_1[0][0]             │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ dense_2 (Dense)              │ (None, 25, 64)            │           8,256 │ dropout_1[0][0]           │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ add_1 (Add)                  │ (None, 25, 64)            │               0 │ layer_normalization[0][0… │
│                              │                           │                 │ dense_2[0][0]             │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ layer_normalization_1        │ (None, 25, 64)            │             128 │ add_1[0][0]               │
│ (LayerNormalization)         │                           │                 │                           │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ multi_head_attention_1       │ (None, 25, 64)            │          16,640 │ layer_normalization_1[0]… │
│ (MultiHeadAttention)         │                           │                 │ layer_normalization_1[0]… │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ add_2 (Add)                  │ (None, 25, 64)            │               0 │ layer_normalization_1[0]… │
│                              │                           │                 │ multi_head_attention_1[0… │
├──────────────────────────────┼───────────────────────────┼─────────────────┼───────────────────────────┤
│ layer_normalization_2        │ (None, 2

 Total params: 124,161 (485.00 KB)

 Trainable params: 124,161 (485.00 KB)

 Non-trainable params: 0 (0.00 B)

**Train 25 epochs** in 5 stages of 5, printing the validation CCC.

In [9]:
for stage in range(5):
    model.fit(X_train, y_train, epochs=5, batch_size=128, verbose=0)
    p = model.predict(X_val, verbose=0).flatten()
    print(f"  epoch {(stage + 1) * 5:3d}  val CCC = {ccc(y_val, p):.3f}")

  epoch   5  val CCC = 0.798


  epoch  10  val CCC = 0.803


  epoch  15  val CCC = 0.839


  epoch  20  val CCC = 0.838


  epoch  25  val CCC = 0.836


**Results** next to the saved result (`reports/transformer_results.csv`).

In [10]:
val_ccc = ccc(y_val, model.predict(X_val, verbose=0).flatten())
pred = model.predict(X_test, verbose=0).flatten()
print(f"parameters {model.count_params():,} | validation CCC {val_ccc:.3f} | TEST CCC {ccc(y_test, pred):.3f} "
      f"| Pearson {np.corrcoef(y_test, pred)[0, 1]:.3f} | MAE {mean_absolute_error(y_test, pred):.3f}")
pd.DataFrame([dict(model="Transformer", params=model.count_params(), val_ccc=val_ccc, test_ccc=ccc(y_test, pred),
                   pearson=np.corrcoef(y_test, pred)[0, 1], mae=mean_absolute_error(y_test, pred))]).to_csv("reports/transformer_results.csv", index=False)
print("saved reports/transformer_results.csv")

parameters 124,161 | validation CCC 0.836 | TEST CCC 0.841 | Pearson 0.870 | MAE 0.060
saved reports/transformer_results.csv
